# Bayes' Nets, Naive Bayes and Perceptrons on Real Data


## Task 0: Prepare data

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

data = load_breast_cancer(as_frame=True)
df = data.frame

# 1. Keep the 3 features plus target
features = ["mean radius", "mean texture", "mean concavity"]
df = df[features + ["target"]]
df.head()

,mean radius,mean texture,mean concavity,target
0,17.99,10.38,0.3001,0
1,20.57,17.77,0.0869,0
2,19.69,21.25,0.1974,0
3,11.42,20.38,0.2414,0
4,20.29,14.34,0.1980,0


In [ ]:
# 2. Train/test split (80% / 20%)
X = df[features]
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)
print("Train size:", len(X_train), "| Test size:", len(X_test))

Train size: 455 | Test size: 114


In [ ]:
# 3. Low/High labels using the TRAIN medians (reused for the test set)
medians = X_train.median()
print(medians)

def discretize(X, medians):
    D = pd.DataFrame(index=X.index)
    for col in X.columns:
        D[col] = np.where(X[col] > medians[col], "High", "Low")
    return D

D_train = discretize(X_train, medians)
D_test = discretize(X_test, medians)

# Readable diagnosis names for the tables
names = {0: "malignant", 1: "benign"}
diag_train = y_train.map(names)
D_train.head()

mean radius       13.30000
mean texture      18.68000
mean concavity     0.06154
dtype: float64


,mean radius,mean texture,mean concavity
68,Low,Low,High
181,High,High,High
63,Low,Low,Low
248,Low,High,Low
60,Low,Low,Low


## Task A: A Small Bayes' Net by Counting

In [ ]:
# 1. CPTs by counting on the training data
p_diag = diag_train.value_counts(normalize=True)
print("P(Diagnosis)")
print(p_diag.round(3), "\n")

cpt = {}
for col in features:
    cpt[col] = pd.crosstab(diag_train, D_train[col], normalize="index")
    print(f"P({col} | Diagnosis)")
    print(cpt[col].round(3), "\n")

P(Diagnosis)
target
benign       0.629
malignant    0.371
Name: proportion, dtype: float64 

P(mean radius | Diagnosis)
mean radius   High    Low
target                   
benign       0.259  0.741
malignant    0.905  0.095 

P(mean texture | Diagnosis)
mean texture   High    Low
target                    
benign        0.322  0.678
malignant     0.799  0.201 

P(mean concavity | Diagnosis)
mean concavity   High    Low
target                      
benign          0.245  0.755
malignant       0.929  0.071 



In [ ]:
# 2. Posterior P(Diagnosis | R, T, C) using the CPTs
def posterior(radius, texture, concavity):
    values = {"mean radius": radius, "mean texture": texture,
              "mean concavity": concavity}
    scores = {}
    for d in ["malignant", "benign"]:
        s = p_diag[d]
        for col in features:
            s *= cpt[col].loc[d, values[col]]
        scores[d] = s
    total = sum(scores.values())          # normalise so the two add up to 1
    return {d: s / total for d, s in scores.items()}

p1 = posterior("High", "High", "High")
p2 = posterior("Low", "Low", "Low")
print("P(malignant | Radius=High, Texture=High, Concavity=High) =",
      round(p1["malignant"], 4))
print("P(malignant | Radius=Low,  Texture=Low,  Concavity=Low)  =",
      round(p2["malignant"], 4))

P(malignant | Radius=High, Texture=High, Concavity=High) = 0.9512
P(malignant | Radius=Low,  Texture=Low,  Concavity=Low)  = 0.0021


In [ ]:
# 3. First test patient: discretized values + CPTs
patient_d = D_test.iloc[0]
print("First test patient (discretized):")
print(patient_d.to_dict())

pa = posterior(patient_d["mean radius"], patient_d["mean texture"],
               patient_d["mean concavity"])
p_malignant_A = pa["malignant"]
print("\nPart A: P(malignant | features) =", round(p_malignant_A, 4))
print("Actual label:", names[y_test.iloc[0]])

First test patient (discretized):
{'mean radius': 'Low', 'mean texture': 'Low', 'mean concavity': 'High'}

Part A: P(malignant | features) = 0.0783
Actual label: benign


## Task B: Naive Bayes

In [ ]:
# 1. GaussianNB on the raw (continuous) features
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix

nb = GaussianNB()
nb.fit(X_train, y_train)

# 2. Accuracy and confusion matrix
y_pred = nb.predict(X_test)
print("Naive Bayes test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(confusion_matrix(y_test, y_pred))

Naive Bayes test accuracy: 0.9561
[[39  4]
 [ 1 70]]


In [ ]:
# 3. predict_proba for the same first test patient (raw values)
patient_raw = X_test.iloc[[0]]
print(patient_raw)

proba = nb.predict_proba(patient_raw)[0]
p_malignant_B = proba[list(nb.classes_).index(0)]   # class 0 = malignant

print("\nPart A (Bayes net, discretized): P(malignant) =", round(p_malignant_A, 4))
print("Part B (GaussianNB, raw values):  P(malignant) =", round(p_malignant_B, 4))

     mean radius  mean texture  mean concavity
204        12.47          18.6         0.08005

Part A (Bayes net, discretized): P(malignant) = 0.0783
Part B (GaussianNB, raw values):  P(malignant) = 0.0358


## Task C: Perceptron

In [ ]:
# 1 & 2. Standardize (fit on train only), train, report accuracy and coefficients
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron

scaler = StandardScaler()
Xs_train = scaler.fit_transform(X_train)
Xs_test = scaler.transform(X_test)

perc = Perceptron(max_iter=1000, random_state=42)
perc.fit(Xs_train, y_train)

print("Perceptron (3 features) test accuracy:",
      round(perc.score(Xs_test, y_test), 4))
print("Coefficients:")
for name, w in zip(features, perc.coef_[0]):
    print(f"  {name:16s} {w:8.3f}")
print("Intercept:", perc.intercept_)

Perceptron (3 features) test accuracy: 0.9211
Coefficients:
  mean radius        -2.176
  mean texture       -1.815
  mean concavity     -1.726
Intercept: [1.]


In [ ]:
# 3. Compare: only mean radius vs all 3 features
results = []
for cols in [["mean radius"], features]:
    sc = StandardScaler()
    Xtr = sc.fit_transform(X_train[cols])
    Xte = sc.transform(X_test[cols])
    p = Perceptron(max_iter=1000, random_state=42).fit(Xtr, y_train)
    results.append({
        "features": ", ".join(cols),
        "test accuracy": round(p.score(Xte, y_test), 4),
        "coefficients": np.round(p.coef_[0], 3).tolist(),
    })
pd.DataFrame(results)

,features,test accuracy,coefficients
0,mean radius,0.8772,[-1.424]
1,"mean radius, mean texture, mean concavity",0.9211,"[-2.176, -1.815, -1.726]"
